### Conversion of google sheets workout data to normalized sql

In [1]:
import pandas as pd

# first I need to clean the google sheets data

# Loading in workouts from google sheets
sheet_id = "1TM8IcrqlYRlk8Ggxig1XMGh5e12KzZTGbh41tp6IB6k"
sheet_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

g_workouts = pd.read_csv(sheet_url)

# Ensure timestamp is in datetime format
g_workouts['Timestamp'] = pd.to_datetime(g_workouts['Timestamp'])

# Trim whitespace from the 'Exercise' column
g_workouts['Exercise'] = g_workouts['Exercise'].str.strip()

# checking list of individual exercises
listers = g_workouts['Exercise'].unique().tolist()

In [2]:
# I don't know how to effectively find workout dates in my google sheets db that aren't in my sql db. Seemed simple but potentially issues with date variables
# So we're going with a manual filter! run above google sheets code first
# Then run the below code to convert g_workouts_filt to sql db format

date_imports = [d.date() for d in pd.to_datetime(['2025-09-04', '2025-09-06'])]

g_workouts['date'] = g_workouts['Timestamp'].dt.date

g_workouts_filt = g_workouts[g_workouts['date'].isin(date_imports)]

g_workouts_filt = g_workouts_filt.drop(columns=['date'])

In [ ]:
from sqlalchemy import create_engine, text
import pandas as pd
import pytz
from datetime import datetime

engine = create_engine("sqlite:///habits.db")

# Using the filterd google workouts df to account for missing workout in sql db
#g_workouts = g_workouts_filt

# read_sql_query simple 
with engine.begin() as conn:

    # Static inputs into habit_entries db
    ## Timestamp of entry (local time to timestamp)
    eastern = pytz.timezone('America/New_York')
    local_time = datetime.now(eastern).strftime('%Y-%m-%d %H:%M:%S')

    ## habit id for workout 
    workout_habit_id = conn.execute(text("SELECT id FROM habits WHERE name = 'workout'")).fetchone()[0]

    for workout in g_workouts.iterrows():
        log_date = datetime.date(workout[1]["Timestamp"])

        result = conn.execute(text("""INSERT INTO habit_entries (habit_id, log_date, timestamp)
                            VALUES (:habit_id, :log_date, :timestamp)"""),
                        {"habit_id": workout_habit_id, "log_date": str(log_date), "timestamp": local_time})
        
        entry_id = result.lastrowid  # Retrieves the unique ID of the row that was just inserted into the habit_entries table

        for i in range(1, 8):
            question = g_workouts.columns[i].lower()  # Get column name by index
            # Doing some modifcation to match habit_answers db
            if question == "exercise":
                question = "workout_type"
            
            if question == "effort level":
                question = "effort"

            if question == "notes:":
                question = "comment"

                answer = workout[1][i]    # Get value by index

                # Funky logic for the last column of my df because I then need to add a 10Rm column immediately after but for the same entry id
                ## insert statement for the notes column
                conn.execute(text("""INSERT INTO habit_answers (entry_id, question, answer)
                VALUES (:entry_id, :question, :answer)"""),
                {"entry_id": entry_id, "question": question, "answer": str(answer)})

                #Static placeholders for the 10RM column
                question_10rm = "10RM"
                answer_10rm = "False"  # Placeholder value for 10RM

                ## extra insert statement for the static 10RM column
                conn.execute(text("""INSERT INTO habit_answers (entry_id, question, answer)
                VALUES (:entry_id, :question, :answer)"""),
                {"entry_id": entry_id, "question": question_10rm, "answer": str(answer_10rm)})

                break

            answer = workout[1][i]    # Get value by index

            conn.execute(text("""INSERT INTO habit_answers (entry_id, question, answer)
                VALUES (:entry_id, :question, :answer)"""),
                {"entry_id": entry_id, "question": question, "answer": str(answer)})

In [ ]:
# workout options function to read in workout choices from the habit_answers db instead of the google sheets
# New workout options function

from sqlalchemy import create_engine, text
import pandas as pd

def load_workout_options():

    engine = create_engine("sqlite:///habits.db")
    
    with engine.connect() as connection:
        workouts = pd.read_sql_query(
            text("""SELECT DISTINCT answer FROM habit_answers
                WHERE question = 'workout_type'"""), connection)
        
    workouts_list = sorted(workouts['answer'].tolist())

    # Add "Other" option to the list of book titles and works. This way other is not saved to the database
    workouts_list.append("Other")

    return workouts_list

workouts_list = load_workout_options()

### Exercise Filter code

In [ ]:
from sqlalchemy import create_engine, text
import pandas as pd
engine = create_engine("sqlite:///habits.db")

columns = ['entry_id', '10RM', 'comment', 'effort', 'reps', 'sets', 'weight', 'workout_type']
workout_df_total = pd.DataFrame(columns=columns)
specific_exercise = "Shoulders db complex"


with engine.connect() as connection:

    entry_ids = pd.read_sql_query(text("SELECT ha.entry_id FROM habit_answers ha WHERE answer = :exercise"), connection, params={"exercise": specific_exercise})

    entry_ids = entry_ids['entry_id'].tolist()


    for id in entry_ids:
        workout_df = pd.read_sql_query(text("SELECT * FROM habit_answers ha WHERE entry_id = :id"), connection, params={"id": id})

        workout_df = workout_df[['entry_id', 'question', 'answer']]

        workout_df = workout_df.pivot(index='entry_id', columns='question', values='answer').reset_index() 

        workout_df_total = pd.concat([workout_df_total, workout_df], ignore_index=True)

    habit_entries = pd.read_sql_query(text("SELECT * FROM habit_entries"), connection)

    workout_df_total = workout_df_total.merge(habit_entries, left_on='entry_id', right_on='id', how='left')

    workout_df_total = workout_df_total[["log_date", "workout_type", "weight", "sets", "reps", "effort", "comment"]]

    workout_df_total['comment'] = workout_df_total['comment'].str.replace("nan", "")

    # Convert the timestamp variable to a datetime format, i'm unsure why I have to do this because that's how it's coded and uploaded in the databse
    workout_df_total['log_date'] = pd.to_datetime(workout_df_total['log_date'])

    # Then I convert that that datetime variable into a more readable string format/ might even decide I don't want to display hours and minutes
    workout_df_total['log_date'] = workout_df_total['log_date'].dt.strftime('%B %d, %Y')

    # Rename columns for clarity
    workout_df_total.columns = ['Timestamp', 'Exercise', 'Weight', 'Sets', 'Reps', 'Effort Level', 'Notes:']

    # Formatting weight variable to show 1 decimal place only if it exits, otherwise I make it a string and strip the '.' and ending '0'
    # Idk how I feel about this as I lose weight as a numeric varaible but should be contained in this function only
    #workout_df_total['weight'] = workout_df_total['weight'].apply(lambda w: f"{w:.1f}".rstrip('0').rstrip('.') if pd.notnull(w) else '')

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo 

server_timestamp = datetime.now(ZoneInfo("New_York")).strftime("%Y-%m-%d %H:%M:%S")
print(server_timestamp)

In [ ]:
# Reading in the json from the sql db
import json
import sqlite3

conn = sqlite3.connect("habits.db")
cursor = conn.cursor()
cursor.execute("SELECT work_list FROM workouts")
result = cursor.fetchone() # only fetches one row from db

workout_list = json.loads(result[0])
print(workout_list)  # results back into list

In [ ]:
# OLD way of Getting all the unqiue book titles from the habit_logs table
import json
from sqlalchemy import create_engine, text
import pandas as pd

engine = create_engine("sqlite:///habits.db")

# read_sql_query simple 
with engine.connect() as connection:
    df_filter = pd.read_sql_query(
        text("""SELECT * FROM habit_logs
             WHERE habit_type = 'reading'"""), connection)

engine.dispose()

# Parse JSON strings into dictionaries
df_parsed = df_filter["data"].apply(json.loads)

# Expand JSON dicts into separate columns
df_expanded = pd.json_normalize(df_parsed)

book_titles = df_expanded["book_title"].unique().tolist()

In [ ]:
# New method to get unique book titel from habit_anwers table
from sqlalchemy import create_engine, text
import pandas as pd

engine = create_engine("sqlite:///habits.db")

# read_sql_query simple 
with engine.connect() as connection:
    books_options = pd.read_sql_query(
        text("""SELECT DISTINCT answer FROM habit_answers
             WHERE question = 'book_title'"""), connection)
    
book_titles = books_options["answer"].to_list()

In [ ]:
# Getting all the unqiue book titles from the habit_logs table
import json
from sqlalchemy import create_engine, text
import pandas as pd

engine = create_engine("sqlite:///habits.db")

# read_sql_query simple 
with engine.connect() as connection:
    workouts = pd.read_sql_query(
        text("""SELECT exercise FROM workouts"""), connection)

engine.dispose()

workouts_list = sorted(workouts["Exercise"].unique().tolist())

In [ ]:
# Cleaning the sql db and saving it back to the same table, specifically clearing leading/trailing whitespace from the 'exercise' column

import pandas as pd
from sqlalchemy import create_engine, text

# Connect to your database
engine = create_engine("sqlite:///habits.db")  # or other dialects like postgresql://...

# Step 1: Read the table into pandas
with engine.connect() as connection:
    df = pd.read_sql_query(text("SELECT * FROM workouts"), connection)

# Step 2: Clean the data (strip leading/trailing whitespace from 'exercise' column)
df['Exercise'] = df['Exercise'].str.strip()

# Step 4: Save it back to the same table (overwrite or update)
df.to_sql("workouts", engine, if_exists="replace", index=False)

### Initial way to create habits sql db

In [ ]:
def init_db():
    metadata = MetaData()

    habit_logs = Table(
        "habit_logs", metadata,
        Column("id", Integer, primary_key=True, autoincrement=True),
        Column("habit_type", String),
        Column("data", Text),
        Column("log_date", Date),
        Column("timestamp", DateTime, server_default=func.current_timestamp()))

    # Create the table if it doesn't exist
    metadata.create_all(engine)
    engine.dispose()

### Final way to create SQL habits db (more normalized structure)

In [ ]:
# initalizing habits websites with sql tables needed to support it
def init_db():
    with engine.begin() as conn:

        # habits db creation
        conn.execute(text("""CREATE TABLE IF NOT EXISTS habits (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            name TEXT UNIQUE NOT NULL);"""))

        # habit_entries db creation 
        conn.execute(text("""CREATE TABLE IF NOT EXISTS habit_entries (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            habit_id INTEGER NOT NULL,
                            log_date DATE NOT NULL,
                            timestamp DATETIME NOT NULL,
                            FOREIGN KEY (habit_id) REFERENCES habits(id));"""))

        # habit_answers db creation 
        conn.execute(text("""CREATE TABLE IF NOT EXISTS habit_answers (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            entry_id INTEGER NOT NULL,
                            question TEXT NOT NULL,
                            answer TEXT NOT NULL,
                            FOREIGN KEY (entry_id) REFERENCES habit_entries(id));"""))
    engine.dispose()

### KPI statistics for static visualization dashbaord

In [ ]:
# YTD Workouts w/ tagerts
# Grabbing monthly exercise and workout statistics for this month and last month to be displayed
import pandas as pd
from sqlalchemy import create_engine, text
import datetime as dt

def get_kpi_stats():

    # Initializing the SQLite database link
    DATABASE = 'habits.db'
    engine = create_engine(f"sqlite:///{DATABASE}")

    # Read workouts data from database
    with engine.connect() as connection:
        workout_df = pd.read_sql_query(text("SELECT * FROM workouts"), connection)

    # Ensure timestamp is in datetime format
    workout_df['Timestamp'] = pd.to_datetime(workout_df['Timestamp'])

    workout_df['month'] = workout_df['Timestamp'].dt.month_name()
    workout_df['year'] = workout_df['Timestamp'].dt.year
    workout_df['date'] = workout_df['Timestamp'].dt.date

    # Get current date and year/month details
    current_date = pd.Timestamp.now()
    current_year = current_date.year
    current_month = current_date.month

    # Calculate Year-to-Date count
    ytd_count = len(workout_df[workout_df['year'] == current_year])

    # Calculate Month-to-Date count
    mtd_count = len(workout_df[(workout_df['year'] == current_year) & 
                        (workout_df['month'] == current_date.month_name())])

    # You can also calculate previous month's total for comparison
    prev_month = current_month - 1 if current_month > 1 else 12
    year = current_year if current_month > 1 else current_year - 1

    prev_month_count = len(workout_df[(workout_df['Timestamp'].dt.year == year) & 
                                    (workout_df['Timestamp'].dt.month == prev_month)])

    # MTD Workouts w/ tagerts

    # Excersies per workout avg

    # days exercised by month
    workouts_by_month_df = workout_df.groupby(['month', 'year']).agg({'Timestamp': 'count'})

    # Days exercised by month
    days_per_month_df = workout_df.groupby(['date', 'month']).agg({'Timestamp': 'count'}).groupby(['month']).agg({'Timestamp': 'count'})

    # Format your DataFrame tables as HTML with styling
    days_per_month_html = days_per_month_df.to_html(classes='workout-table', index=True, border=0)
    workouts_by_month_html = workouts_by_month_df.to_html(classes='workout-table', index=True, border=0)

    return ytd_count, mtd_count, prev_month_count, days_per_month_html, workouts_by_month_html

ytd_count, mtd_count, prev_month_count, days_per_month_html, workouts_by_month_html = get_kpi_stats()

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
import datetime as dt

# Initializing the SQLite database link
DATABASE = 'habits.db'
engine = create_engine(f"sqlite:///{DATABASE}")

# Read workouts data from database
with engine.connect() as connection:
    workout_df = pd.read_sql_query(text("SELECT * FROM workouts"), connection)

# Ensure timestamp is in datetime format
workout_df['Timestamp'] = pd.to_datetime(workout_df['Timestamp'])

workout_df['month'] = workout_df['Timestamp'].dt.month_name()
workout_df['year'] = workout_df['Timestamp'].dt.year
workout_df['date'] = workout_df['Timestamp'].dt.date

# Get current date and year/month details
current_date = pd.Timestamp.now()
current_year = current_date.year
current_month = current_date.month

# Calculate Year-to-Date count
ytd_count = len(workout_df[workout_df['year'] == current_year])

# Calculate Month-to-Date count
mtd_count = len(workout_df[(workout_df['year'] == current_year) & 
                        (workout_df['month'] == current_date.month_name())])


### Gathering timestamp of habit entry

In [ ]:
from datetime import datetime
import pytz

# Use America/New_York for EST/EDT (automatically adjusts for DST)
eastern = pytz.timezone('America/New_York')
local_time = datetime.now(eastern).strftime('%Y-%m-%d %H:%M:%S')

## Working on 10RM data output

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

# Loading in workout options to then add 10RM suffix
def load_workout_options():
    engine = create_engine(f"sqlite:///habits.db")

    # Gathering all unique exercise names (later will grab just workout names with 10Rm variable) 
    with engine.connect() as connection:
        workouts = pd.read_sql_query(
            text("""select distinct ha.answer from habit_answers ha 
                    left join habit_entries he 
                    on he.id = ha.entry_id 
                    where he.habit_id = 3 and ha.question = 'workout_type' 
                    order by ha.answer"""), connection)

    engine.dispose()

    workouts.rename(columns={"answer": "Exercise"}, inplace=True)

    return workouts

workouts = load_workout_options()

# Add the "10RM" suffix to each exercise name (observation)
workouts["Exercise"] = workouts["Exercise"] + " 10RM"

# Adding a fake temp column
workouts["weight"] = 100

# Adding spice/variation to the weight variable
workouts.loc[workouts['Exercise'].str.contains("chest", case=False, na=False), 'weight'] = 50

# Adds suffix to row or variable names, not observations
#workouts = workouts.add_prefix("10RM")

In [2]:
import pytz
from datetime import datetime

# Now that I have a list of 10RM workouts, I want to multiply them by the specific percentage over each week 
multipliers = {"Week 1": 0.25,
    "Week 2": 0.30,
    "Week 3": 0.35,
    "Week 4": 0.40,
    "Week 5": 0.60,
    "Week 6": 0.65,
    "Week 7": 0.70,
    "Week 8": 0.75,
    "Week 9": 0.85,
    "Week 10": 0.90,
    "Week 11": 0.95,
    "Week 12": 1}

eastern = pytz.timezone("America/New_York")
date = datetime.now(eastern).strftime("%Y-%m-%d")

rm10_plan = pd.DataFrame(columns=["week_number", "exercise_name", "target_weight", "created_date"])

for activity in workouts.itertuples(index=False):
    activity_name = activity[0]
    activity_weight = activity[1]

    for i, row in enumerate(multipliers.items()):
        week = row[0] 
        multiplier = row[1]
        
        new_weight = activity_weight * multiplier

        #Temporary dictionay to store the data 
        tempdict = {
        "week_number": week,
        "exercise_name": activity_name,
        "target_weight": new_weight,
        "created_date": date} 

        tempdf = pd.DataFrame(tempdict, index=[0])

        rm10_plan = pd.concat([rm10_plan, tempdf], ignore_index=True)

C:\Users\ryani\AppData\Local\Temp\ipykernel_8360\2765287832.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  rm10_plan = pd.concat([rm10_plan, tempdf], ignore_index=True)


In [3]:
# Apply reps and sets based on the week number
import numpy as np

def assign_reps_sets(week):
    if week in ["Week 1", "Week 2", "Week 3", "Week 4"]:
        return 3, "30/25/20"
    elif week in ["Week 5", "Week 6", "Week 7", "Week 8"]:
        return 3, 10
    elif week in ["Week 9", "Week 10", "Week 11", "Week 12"]:
        return 3, "5/3/1"
    else:
        return None, None

rm10_plan[['sets', 'reps']] = rm10_plan['week_number'].apply(lambda x: pd.Series(assign_reps_sets(x)))

In [4]:
# applying workout type to newly created 10RM plan
import numpy as np

rm10_plan["workout_type"] = np.where(rm10_plan["exercise_name"].str.contains("Chest|Triceps", regex = True), "Push", 
                                    np.where(rm10_plan["exercise_name"].str.contains("Back|Biceps", regex = True), "Pull", 
                                             np.where(rm10_plan["exercise_name"].str.contains("Hip|Leg|Calf", regex = True), "Legs", "unknown")))



In [5]:
# Strip string from week number
rm10_plan["week_number"] = rm10_plan["week_number"].str.replace("Week ", "").astype(int)

In [6]:
engine = create_engine(f"sqlite:///habits.db")
for activity in rm10_plan.itertuples(index=False):
    week_number = activity[0] 
    exercise_name = activity[1] 
    target_weight = activity[2] 
    created_date = activity[3] 
    sets = activity[4]
    reps = activity[5]
    workout_type = activity[6]

    with engine.begin() as conn:
        conn.execute(text("""INSERT INTO tenrm_plans (week_number, exercise_name, target_weight, created_date , sets, reps, workout_type)
        VALUES (:week_number, :exercise_name, :target_weight, :created_date , :sets, :reps, :workout_type)"""),
        {"week_number": week_number, "exercise_name": exercise_name, "target_weight": target_weight, "created_date": created_date, "sets": sets, "reps": reps, "workout_type": workout_type})

IntegrityError: (sqlite3.IntegrityError) UNIQUE constraint failed: tenrm_plans.workout_type, tenrm_plans.week_number, tenrm_plans.exercise_name
[SQL: INSERT INTO tenrm_plans (week_number, exercise_name, target_weight, created_date , sets, reps, workout_type)
        VALUES (?, ?, ?, ? , ?, ?, ?)]
[parameters: (1, 'Back ISO rear deltoid machine 10RM', 25.0, '2025-10-25', 3, '30/25/20', 'Pull')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

In [ ]:
activity

# Merge workout data from google sheets to sql db

In [ ]:
##### WHEN READING IN UPDATED DATA FROM GOOGLE SHEET MAKE SURE TO RUN SQL COMMAND TO TRIM WHITESPACES FROM EXERCISE COLUMN ######

# Merge exisiting workouts google sheet to sql db

import pandas as pd
from sqlalchemy import create_engine
import pandas as pd
import os

sheet_id = "1TM8IcrqlYRlk8Ggxig1XMGh5e12KzZTGbh41tp6IB6k"
sheet_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

workouts = pd.read_csv(sheet_url)

# Ensure timestamp is in datetime format
workouts['Timestamp'] = pd.to_datetime(workouts['Timestamp'])

def df_to_sqlite(df: pd.DataFrame, name: str, db_path: str) -> None:
    # Input validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame")
    if not isinstance(name, str):
        raise TypeError("name must be a string")
    if not isinstance(db_path, str):
        raise TypeError("db_path must be a string")

    # Create a SQLite connection string
    engine = create_engine(f"sqlite:///{db_path}")

    # Save the DataFrame to the SQLite database
    df.to_sql(name=name, con=engine, if_exists='replace', index=True) # This is where index is coming from when I create the table

    # Dispose the engine
    engine.dispose()


#df_to_sqlite(workouts, "workouts", "habits.db")

# Apple's health data

## Data cleaning of raw apple health data into cleaned apple workout data

### Create apple_data_raw table initially 

In [ ]:
# Creating the new table in the SQLite database (with I think appropriate data types)

import pandas as pd
from sqlalchemy import create_engine, text

# Connect to SQLite database
engine = create_engine("sqlite:///habits.db")

# Define CREATE TABLE SQL statement
create_table_sql = """
CREATE TABLE IF NOT EXISTS apple_data_raw (
    type TEXT,
    sourceName TEXT,
    value FLOAT,
    unit TEXT,
    startDate TIMESTAMP,
    endDate TIMESTAMP,
    creationDate TIMESTAMP,
    sourceVersion TEXT,
    appleStandHours INTEGER,
    appleExerciseTimeGoal INTEGER,
    bpm INTEGER,
    maximum FLOAT,
    sum FLOAT,
    appleMoveTimeGoal INTEGER,
    average FLOAT,
    time TIMESTAMP,
    key TEXT,
    duration FLOAT,
    dateComponents TEXT,  -- SQLite doesn't support JSONB, use TEXT or JSON (in newer versions)
    CardioFitnessMedicationsUse BOOLEAN,
    activeEnergyBurned FLOAT,
    appleMoveTime INTEGER,
    date DATE,
    activeEnergyBurnedUnit TEXT,
    locale TEXT,
    appleStandHoursGoal INTEGER,
    BiologicalSex TEXT,
    FitzpatrickSkinType TEXT,
    BloodType TEXT,
    workoutActivityType TEXT,
    minimum FLOAT,
    path TEXT,
    appleExerciseTime INTEGER,
    durationUnit TEXT,
    DateOfBirth DATE,
    device TEXT,
    activeEnergyBurnedGoal INTEGER);"""

# Execute the CREATE TABLE query
with engine.connect() as connection:
    connection.execute(text(create_table_sql))

### .py code for reading in correct csv file

In [ ]:
import os
import datetime as dt
import sys
import pandas as pd

# Defining the subddirectroy 
directory = "apple/"

# Grabbing today's date
today = dt.datetime.now().strftime('%Y-%m-%d')

# Getting user input at the breginning of this script to ensure I'm grabbing the correct file
while True:
    choice = input("Are you using today's date to uploaded apple data? (y/n) ").lower()
    if choice == 'y':
        # Lopping through each of the files in the directory to find a csv file with today's date
        for file in os.listdir(directory):
            if file.endswith(".csv") and today in file:
                apple_file = file
        break
    if choice == "n":
        sys.exit()

### Appending new apple workout data to existing apple_data_raw table

In [ ]:
# Ensure I'm only adding new data to the sql db
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(f"sqlite:///habits.db")

# Read from SQLite and parse those columns as datetime
with engine.connect() as connection:
    max_date_db = pd.read_sql_query(
        text("SELECT max(StartDate) FROM apple_data_raw"), 
        connection,
        parse_dates=['startDate'])   
engine.dispose()

# Grabbing the max date from dataset to filter new data off of  
max_date_db = max_date_db['max(StartDate)'].tolist()[0]

In [ ]:
# Read in new CSV File
apple = pd.read_csv(r"apple/apple_health_export_2025-10-10.csv")

### Cleaning Data ###

# Remove observations from csv file that are already in my sql db (remove this line if starting from scratch)
apple = apple[apple["startDate"] >= max_date_db]

# Fitler the source to only my Apple Watch (Could pull in other devices later like iPhone or gamin)
apple = apple[(apple['sourceName'] == "Ryan’s Apple\xa0Watch") | (apple['sourceName'].isna())]

# Strip the "HKWorkoutActivityType" prefix from the workout types
apple["workoutActivityType"] = apple["workoutActivityType"].str.replace("HKWorkoutActivityType", "")

# Convert date columns to datetime format (took 16 min to run on 2.4 million rows)
datetime_cols = ['startDate', 'endDate', 'creationDate']
for col in datetime_cols:
    apple[col] = pd.to_datetime(apple[col], errors='coerce', utc=True)

for col in datetime_cols:
    apple[col] = apple[col].dt.strftime('%Y-%m-%dT%H:%M:%S%z')

# Drop the UUID column, apple added this since last 7/18 upload
apple.drop(columns=['uuid'], inplace=True)

In [ ]:
# Appending the new raw data to my sql lite table
apple.to_sql(name="apple_data_raw", con=engine, if_exists='append', index=False)

### Clean apple_data_raw and upload to cleaned apple_workouts table

#### Intially create apple_workouts table

In [ ]:
# Creating the new table in the SQLite database (with I think appropriate data types)

import pandas as pd
from sqlalchemy import create_engine, text

# Connect to SQLite database
engine = create_engine("sqlite:///habits.db")

# Define new CREATE TABLE SQL statement based on your dataframe columns
create_table_sql = """CREATE TABLE IF NOT EXISTS apple_workouts (
    StartDate TIMESTAMP,
    activity TEXT,
    metric TEXT,
    measurement_type TEXT,
    value FLOAT,
    d_unit TEXT,
    EndDate TIMESTAMP,
    workout_id TEXT,
    week_period DATE,
    month TEXT,
    activity_type TEXT);"""


# Execute the CREATE TABLE query
with engine.connect() as connection:
    connection.execute(text(create_table_sql))

### Cleaning raw apple workout for new table (apple workouts)  

In [ ]:
# Read in raw apple workouts data from the sql db
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

def Read_Apple_Workouts():
    engine = create_engine(f"sqlite:///habits.db")

    # Read from SQLite and parse those columns as datetime
    with engine.connect() as connection:
        apple = pd.read_sql_query(
            text("SELECT * FROM apple_data_raw"),
            connection,
            parse_dates=['startDate', 'endDate', 'creationDate'])
        
    engine.dispose()

    # Using max date from the inital upload into the apple workouts raw db to also ensure only new observations are going into the apple workouts table 
    apple = apple[apple["startDate"] >= max_date_db]

    # Gather a df of workouts types, their dates, and key statistics
    # Filters df to include only rows where at least one of the workout-related columns is not missing (NaN).
    # .notna() Check for Non-Nulls in selected columns 
    # .any(axis=1) aggregates rows where there's at least one non-missing value 
    apple_workouts = apple[apple[['maximum', 'minimum', 'sum', 'average', 'duration', 'workoutActivityType', 'durationUnit']].notna().any(axis=1)] 

    # Select particular variables
    apple_workouts = apple_workouts[["workoutActivityType", "startDate", "endDate", "type", "maximum", "minimum", "sum", "average", "duration", "durationUnit"]]

    return apple_workouts

apple_workouts = Read_Apple_Workouts()

In [ ]:
# Convert apple workouts to long format
def aw_to_long(apple_workouts):

    # Reshape the apple_workouts DataFrame from wide format (one row per workout with multiple measurement columns) to long format (one row per measurement per workout).
    df_long = pd.melt(apple_workouts, 
                        id_vars=['startDate', 'endDate', 'workoutActivityType', 'type', 'duration', 'durationUnit'],
                        value_vars=['maximum', 'minimum', 'sum', 'average'], # turned into rows via where the name goes into 'measurement_type' and actual values to the 'value' column
                        var_name='measurement_type', 
                        value_name='value')

    # rRemove rows where both type and value are null (no meaningful data here)
    df_long = df_long[(df_long['type'].notna()) | (df_long['value'].notna())]

    # Handle duration rows separately and combine
    duration_rows = apple_workouts[apple_workouts['duration'].notna() & apple_workouts['type'].isna()].copy()
    duration_rows = duration_rows.assign(
        measurement_type='total',
        value=duration_rows['duration'],
        type='Duration')[['startDate', 'endDate', 'workoutActivityType', 'type', 'measurement_type', 'value', 'durationUnit']]

    # Combine and clean
    result = pd.concat([df_long[df_long['value'].notna()][['startDate', 'workoutActivityType', 'type', 'measurement_type', 'value', 'durationUnit']],
        duration_rows], ignore_index=True)

    # Rename columns for clarity
    result.columns = ['StartDate', 'activity', 'metric', 'measurement_type', 'value', 'd_unit', 'EndDate']

    # cleaning value column to only go out 2 decimal places
    result['value'] = result['value'].round(2)

    result.sort_values('StartDate', ascending=False, inplace=True)

    return result

aw_long = aw_to_long(apple_workouts)

In [ ]:
# Adding activity id

def add_workout_id(aw_long):
    # Identify rows where activity is not null (these rows define the actual activity)
    activity_map = aw_long[aw_long['activity'].notnull()]

    activity_map = activity_map[["StartDate", "activity"]]

    # Creating a workout identifier
    # Creates sequential IDs starting from 0
    # Convert index IDs to integers
    activity_map = activity_map.reset_index(drop=True)
    activity_map['workout_id'] = activity_map.index

    # Merge activity back onto the main dataframe using startDate
    aw_final = aw_long.merge(activity_map, on='StartDate', how='left', suffixes=('', '_Specifier'))

    # Forward-fill the activity only for rows with the same startDate (simplicity but not needed )
    aw_final['activity'] = aw_final['activity'].fillna(aw_final['activity_Specifier'])

    # Drop helper column
    aw_final.drop(columns=['activity_Specifier'], inplace=True)

    # Think about later, can't convert column to int because of missing variables that are uploaded from garmin that wasn't filtered out initally because I need blanks
    #aw_final_new = aw_final_new['id'].astype('int')

    # Grabbing just the start of the week in dt fashion
    aw_final['week_period'] = aw_final['StartDate'].dt.to_period('W').apply(lambda r: r.start_time)

    # Adding week periods which is EXACTLY what I want! Idk why I then convert it to string
    #aw_final['week_period'] = aw_final['StartDate'].dt.to_period('W').astype(str)

    # Extract Year-Month for grouping
    aw_final['month'] = aw_final['StartDate'].dt.to_period('M')

    # Adding new activity type variable for minutes calculation 
    aw_final['activity_type'] = np.where(aw_final['activity'].isin(['Running', 'Cycling', 'Swimming']), 'Cardio',
        np.where(aw_final['activity'] == 'TraditionalStrengthTraining', 'Weights', None))

    return aw_final

aw_final = add_workout_id(aw_long)

In [ ]:
# need to do a couple of string conversions before uploading the data to the sql lite db

aw_final['month'] = aw_final['month'].astype(str)
aw_final['workout_id'] = aw_final['workout_id'].astype('Int64').astype(str)
aw_final['StartDate'] = aw_final['StartDate'].dt.tz_convert(None)
aw_final['EndDate'] = aw_final['EndDate'].dt.tz_convert(None)

#### Uploading modified aw_final to the sql db

In [ ]:
engine = create_engine("sqlite:///habits.db")
with engine.connect() as connection:
    aw_final.to_sql(name="apple_workouts", con=engine, if_exists='append', index=False)

# Reading in clean apple health data from sql db

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

# Connect to your SQLite database
engine = create_engine("sqlite:///habits.db")

# Read in the data and parse datetime columns
with engine.connect() as connection:
    aw_final = pd.read_sql_query(
        text("SELECT * FROM apple_workouts"),
        connection,
        parse_dates=['StartDate', 'EndDate', 'week_period']
    )

# Clean up connection
engine.dispose()

# Convert 'month' to a period type for better handling of monthly data
aw_final['month'] = aw_final['month'].astype('period[M]')

## Analysis

### Tabular/Numeric Analysis

In [ ]:
# Number of unqiue activities
session_counts = aw_final[aw_final['metric'].str.contains('Duration')].groupby('activity')['StartDate'].nunique().reset_index(name='session_count')
print(session_counts)

In [ ]:
# Time I spent on each exercise every week
time_week = (
    aw_final[(aw_final['metric'] == "Duration")] # Got ride of activity filter(aw_final['activity'].isin(['Running', 'Cycling'])) 
    .groupby(['activity', 'week_period'])['value']
    .agg(Time='mean', n='count')  # compute both mean and count
    .reset_index())
time_week

In [ ]:
#Miles per week grouped by exerise 

miles_week = (aw_final[(aw_final['activity'].isin(['Running', 'Cycling'])) & (aw_final['metric'].str.contains("Distance"))]
    .groupby(['activity', 'week_period'])['value']
    .agg(Miles='mean', n='count')  # compute both mean and count
    .round(2) # Round to 2 decimal places
    .reset_index())

miles_week

### Graphical Analysis

In [ ]:
# Function that fills in missing data for each grouped by df 
from itertools import product

def fill_missing_combinations(
    original_df,
    aggregated_df,
    time_col,
    category_col,
    value_cols,
    time_filter=None,
    category_values=None):

    # Step 1: Filter time periods if needed
    if time_filter:
        time_values = original_df[time_filter(original_df)].loc[:, time_col].unique()
    else:
        time_values = original_df[time_col].unique()

    # Step 2: Get all categories
    if category_values:
        category_values = category_values
    else:
        category_values = aggregated_df[category_col].unique()

    # Step 3: Create full index
    full_index = pd.DataFrame(
        product(category_values, time_values),
        columns=[category_col, time_col])

    # Step 4: Merge with aggregated data
    filled_df = pd.merge(full_index, aggregated_df, on=[category_col, time_col], how='left')

    # Step 5: Fill NaNs with 0 for specified value columns
    for col in value_cols:
        filled_df[col] = filled_df[col].fillna(0)

    return filled_df.sort_values(by=[time_col, category_col]).reset_index(drop=True)

#### Frequency bar chart grouped by exercise type 

In [ ]:
# Group by month and activity, count sessions
activities = ["Walking", "Cycling", "TraditionalStrengthTraining", "Running", "Swimming"]

# Grab last 5 Months of data
today = pd.Timestamp.today()
l_5_m = (today - pd.DateOffset(months=5)).to_period('M')

df_counts = (aw_final[(aw_final['metric'] == 'Duration') & (aw_final['activity'].isin(activities)) & (aw_final['month'] > l_5_m)]
    .groupby(['month', 'activity'])
    .size()
    .reset_index(name='n'))

In [ ]:
full_count_data = fill_missing_combinations(
    original_df=df_counts,
    aggregated_df=df_counts,
    time_col='month',
    category_col='activity',
    value_cols=['n'])

full_count_data['n'] = full_count_data['n'].astype(int)  # Convert to int for better readability

# Create a string label for display
full_count_data['month_label'] = full_count_data['month'].dt.strftime('%b %Y')

# Set 'month_label' as a categorical(factor variable) with order based on 'month_period'
month_order = full_count_data.sort_values('month')['month_label'].unique()
full_count_data['month_label'] = pd.Categorical(full_count_data['month_label'], categories=month_order, ordered=True)

# Adding a label for 'TraditionalStrengthTraining' top shorten it for the graph output
full_count_data['activity'] = full_count_data['activity'].replace({'TraditionalStrengthTraining': 'Weights'})

In [ ]:
# Set monthly goals per activity (you can make this dynamic)
activity_goals = {'Running': 8, 'Cycling': 8}

# Map the goal into the counts DataFrame
full_count_data['goal'] = full_count_data['activity'].map(activity_goals)

In [ ]:
# Plot
from plotnine import *

plot = (ggplot(full_count_data, aes(x='month_label', y='n', fill='activity')) +
    geom_bar(stat='identity', position='dodge', color = "Black") +
    geom_text(aes(label='n'), position=position_dodge(width=0.9), va='bottom') + # va & ha are used for veritcal and horizontal allignment 
    
    scale_fill_brewer(type='qual', palette='Set2') +
    #scale_color_manual(values={'Running': 'Black', 'Cycling': 'Gray'}) +
    scale_y_continuous(breaks = range(0, 16),
                       limits = [0, 15]) +

    labs(title='Workout Frequency by Month and Activity',
        x='',
        y='Sessions',
        fill='Activity') +
    #theme_matplotlib() +
    theme_seaborn() +
    theme(figure_size=(10, 5)))

plot

#### Distance per week grouped by exercise type

In [ ]:
miles_week = (aw_final[(aw_final['activity'].isin(['Running', 'Cycling'])) & (aw_final['metric'].str.contains("Distance")) & (aw_final['month'] > "2025-03")]
    .groupby(['activity', 'week_period'])['value']
    .agg(Total_Miles='sum', n='count')  # compute both mean and count
    .round(2) # Round to 2 decimal places
    .reset_index())

In [ ]:
full_miles_week = fill_missing_combinations(
    original_df=aw_final,
    aggregated_df=miles_week,
    time_col='week_period',
    category_col='activity',
    value_cols=['Total_Miles', 'n'],
    time_filter=lambda df: df['month'] > "2025-03",
    category_values=['Running', 'Cycling'])


# Create a string label for display
full_miles_week['week_label'] = full_miles_week['week_period'].dt.strftime('%b %d')

# Set 'week_label' as a categorical(factor variable) with order based on 'week_period'
week_order = full_miles_week.sort_values('week_period')['week_label'].unique()
full_miles_week['week_label'] = pd.Categorical(full_miles_week['week_label'], categories=week_order, ordered=True)

In [ ]:
from plotnine import *

plot = (ggplot(full_miles_week, aes(x='week_label', y='Total_Miles', fill='activity')) +
    geom_bar(stat='identity', position='dodge', color = "Black") +
    geom_text(aes(label='n'), position=position_dodge(width=.9), va='bottom') +

    scale_y_continuous(breaks = range(0, 36, 5),
                       #minor_breaks=range(0, 36, 2),  # Can't do minor ticks 2.5 because int not float :(
                       limits = [0, 36]) +

    scale_fill_manual(values={'Running': '#a259d9',   
        'Cycling': '#ff9800'}) +

    labs(title= "Cardio Miles per Week",
        x="",
        y="Miles",
        fill = "Activity") +
    theme_seaborn() +
    theme(figure_size=(10, 5),
        panel_grid_minor_y = element_line(color = "White", linetype = "dotted")))

plot

#### Time spent cardio & weight lifting per week with goals 

In [ ]:
mins_week = (aw_final[
        (aw_final['activity_type'].notna()) &
        (aw_final['metric'] == "Duration") &
        (aw_final['month'] > "2025-03")]
    .groupby(['activity_type', 'week_period'])['value']
    .agg(Total_min='sum', n='count')  # compute sum and count
    .round(2)  # Round to 2 decimal places
    .reset_index())

In [ ]:
full_mins_week = fill_missing_combinations(
    original_df=aw_final,
    aggregated_df=mins_week,
    time_col='week_period',
    category_col='activity_type',
    value_cols=['Total_min', 'n'],
    time_filter=lambda df: df['month'] > "2025-03",
    category_values=['Cardio', 'Weights'])

# Create a string label for display
full_mins_week['week_label'] = full_mins_week['week_period'].dt.strftime('%b %d')

# Set 'week_label' as a categorical(factor variable) with order based on 'week_period'
week_order = full_mins_week.sort_values('week_period')['week_label'].unique()
full_mins_week['week_label'] = pd.Categorical(full_mins_week['week_label'], categories=week_order, ordered=True)

In [ ]:
# Goals 
# 150 minutes of moderate aerobic activity & 2 strength training exercises

from plotnine import *
plot = (ggplot(full_mins_week, aes(x='week_label', y='Total_min', fill='activity_type')) +
    geom_bar(stat='identity', position='dodge', color = "Black") +
    geom_text(aes(label='Total_min'), format_string='{:.1f}',position=position_dodge(width=.9), va='bottom') + #Format count?

    geom_hline(yintercept=150, color="#549f74", linetype='dashed', size=1) + # Cardio goal
    geom_hline(yintercept=80, color="#b36a62", linetype='dashed', size=1) + # Weights goal
    
    scale_y_continuous(breaks = range(0, 450, 50),
                       #minor_breaks=range(0, 36, 2),  # Can't do minor ticks 2.5 because int not float :(
                       limits = [0, 400]) +

    # Manual color scales
    scale_fill_manual(values={'Cardio': '#52be80', 'Weights': '#ec7063'}) +  # Bar colors
    
    labs(title= "Minutes per Week by Activity",
         x="",
         y="Minutes",
         fill="Activity") +
     
    theme_seaborn() +
    theme(figure_size=(10, 5),
        panel_grid_minor_y = element_line(color = "White", linetype = "dotted")))
plot

#### Time spent working out per week with count

In [ ]:
workout_time = aw_final[(aw_final['StartDate'].dt.year >= 2025) & 
                        (aw_final['metric'] == 'Duration')].groupby(['week_period'])['value'].agg(Time='sum', n='count').reset_index()

workout_time['Time'] = workout_time['Time'].round(2)

In [ ]:
from plotnine import *

plot = (ggplot(workout_time, aes(x='week_period', y='Time')) +
     geom_line(color = "blue", size = 1) +
     geom_point(aes(size = "n"), alpha = 0.6, color = "blue") +
    geom_text(aes(label='n'), format_string='{:.0f}', va='bottom') +
     labs(title= "Minutes per Week by Activity",
         x="Week",
         y="Minutes") +
     scale_x_datetime(date_labels='%b %d', date_breaks='1 week') +
     
    theme_seaborn() +
    theme(panel_grid_minor_y = element_line(color = "gray", linetype = "dotted"),
        figure_size=(10, 5),
        axis_text_x=element_text(angle=25, hjust=1),
        legend_position='none',
        axis_ticks_minor_x=element_blank()))
plot

#### Step Count Boxplot

In [ ]:
def Read_Apple_Workouts():
    engine = create_engine(f"sqlite:///habits.db")

    # Read from SQLite and parse those columns as datetime
    with engine.connect() as connection:
        apple = pd.read_sql_query(
            text("""SELECT type, value, startDate, endDate 
                FROM apple_data_raw
                WHERE type = 'StepCount' AND value IS NOT NULL and startDate >= '2025-01-01'
                ORDER BY startDate DESC"""), connection,
            parse_dates=['startDate', 'endDate'])
    engine.dispose()

    return apple

apple = Read_Apple_Workouts()

# Group by date (not datetime)
steps_day = apple.groupby(apple['startDate'].dt.date)['value'].agg(steps='sum', n='count').reset_index()

# Convert 'startDate' to datetime
steps_day['startDate'] = pd.to_datetime(steps_day['startDate'])

# Extract month as a period for correct ordering
steps_day['month_period'] = steps_day['startDate'].dt.to_period('M')

# Create a string label for display
steps_day['month_label'] = steps_day['month_period'].dt.strftime('%b %Y')

# Set 'month_label' as a categorical(factor variable) with order based on 'month_period'
month_order = steps_day.sort_values('month_period')['month_label'].unique()
steps_day['month_label'] = pd.Categorical(steps_day['month_label'], categories=month_order, ordered=True)

In [ ]:
from plotnine import *

plot = (
    ggplot(steps_day, aes(x='month_label', y='steps', fill='month_label')) +

    geom_boxplot(color="black") +
    stat_summary(fun_data="mean_cl_boot", geom = "point", fill = "white", color = "red") +

    labs(title="Steps per Day grouped by Month", 
         x="", 
         y="Steps") +

    scale_y_continuous(breaks = range(0, 22500, 2500),
                    #minor_breaks=range(0, 36, 2),  # Can't do minor ticks 2.5 because int not float :(
                    limits = [0, 20000]) +

    theme_seaborn() +

    theme(axis_text_x=element_text(),
        figure_size=(10, 5),
        legend_position='none',
        panel_grid_minor_y = element_line(color = "White", linetype = "dotted")))
plot

#### Area chart attempt :/ 

In [ ]:
# Group by month and activity, count sessions
activities = ["Walking", "Cycling", "TraditionalStrengthTraining", "Running", "Swimming"]

workout_time = aw_final[(aw_final['StartDate'].dt.year >= 2025) & 
                        (aw_final['metric'] == 'Duration') &
                        (aw_final['activity'].isin(activities))].groupby(["activity", "month"])['value'].agg(Time='sum', n='count').reset_index()

workout_time['Time'] = workout_time['Time'].round(2)
# Convert your month column to datetime first
workout_time['month'] = workout_time['month'].dt.to_timestamp()


In [ ]:
# Plot
from plotnine import *

plot = (ggplot(workout_time, aes(x='month', y='Time', fill='activity')) +
    geom_bar(stat='identity', position='dodge') +
    geom_text(aes(label='n'), position=position_dodge(width=0.9), va='bottom') + # va & ha are used for veritcal and horizontal allignment 
    #geom_hline(aes(yintercept='goal', color='activity'), linetype='dashed') +
    scale_fill_brewer(type='qual', palette='Set1') +
    #scale_color_manual(values={'Running': 'black', 'Cycling': 'gray'}) +
    #scale_y_continuous(breaks = range(0, 16),
    #                   limits = [0, 15]) +  

    labs(title='Workout Frequency by Month and Activity',
        x='Month',
        y='Number of Sessions',
        fill='Activity',
        color='Goal') +
    #theme_matplotlib() +
    theme_seaborn() +
    theme(figure_size=(10, 5)))

plot

#### Tree Map chart for workout distribution

In [ ]:
import plotly.express as px

activites = ['Running', 'Cycling', 'TraditionalStrengthTraining', 'Swimming']

# Time I spent on each exercise every week

today = pd.Timestamp.today() # IDk how this number doesn't match what's showing on my webpage

five_week_ago = (today - pd.DateOffset(weeks=5)).normalize() # Normalize sets the time to midnight

activity_distribution = (aw_final[
        (aw_final['metric'] == "Duration") & 
        (aw_final['activity'].isin(activites)) & 
        (aw_final['StartDate'] >= five_week_ago)]
     .sort_values(by='StartDate', ascending=False)
    .groupby(['activity'])['value']
    .agg(count='count')
    .reset_index())
             
# Add percent of total column
total = activity_distribution['count'].sum()
activity_distribution['percent'] = activity_distribution['count'] / total

# Format labels as "Activity<br>Count (Percent)"
activity_distribution['label'] = activity_distribution.apply(lambda row: f"{row['activity']}<br>{row['count']} ({row['percent']:.1%})", axis=1)

In [ ]:
# Create treemap
fig = px.treemap(
    activity_distribution,
    path=['label'],         # use custom label for full text
    values='count',
    color='count',
    color_continuous_scale='Blues')

fig.update_traces(
    hovertemplate='%{label}')  

fig.update_layout(showlegend=False, coloraxis_showscale=False)

# Optional: save the plot
fig.show() 

### Plotly interactive visualization

In [ ]:
import plotly.graph_objs as go
import plotly.io as pio

def generate_plotly():
    dates = pd.date_range(end=datetime.today(), periods=30)
    habit_counts = np.random.randint(0, 5, size=30)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=dates,
        y=habit_counts,
        mode='lines+markers',
        name='Habit Check-ins',
        line=dict(color='royalblue', width=2),
        marker=dict(size=6)
    ))
    fig.update_layout(
        title='Interactive Plot: Habit Check-ins Over 30 Days',
        xaxis_title='Date',
        yaxis_title='Check-ins',
        template='plotly_white',
        autosize=True,
        height=500
    ) 
    plotly_html = pio.to_html(fig, full_html=False)

    return plotly_html

In [ ]:
# HTML for plotly graph    
"""     <div class="section">
        <h2>Interactive Workout Analysis</h2>
        {{ plotly_html | safe }}
    </div> """

### HTML Pandas Tables

In [ ]:
######### Depreciated #########

# Initializing the SQLite database link

# Database configuration details - IDK if this is even needed?
def create_sql_engine():
    # Create a SQLAlchemy engine to connect to the SQLite database
    Database = "habits.db" 
    engine = create_engine(f"sqlite:///{Database}")
    return engine

engine = create_sql_engine()

# Read workouts data from database
with engine.connect() as connection:
    workout_df = pd.read_sql_query(text("SELECT * FROM workouts"), connection)

# Ensure timestamp is in datetime format
workout_df['Timestamp'] = pd.to_datetime(workout_df['Timestamp'])

workout_df['month'] = workout_df['Timestamp'].dt.month_name()
workout_df['year'] = workout_df['Timestamp'].dt.year
workout_df['date'] = workout_df['Timestamp'].dt.date

# days exercised by month
workouts_by_month_df = workout_df.groupby(['month', 'year']).agg({'Timestamp': 'count'})

# Days exercised by month
days_per_month_df = workout_df.groupby(['date', 'month']).agg({'Timestamp': 'count'}).groupby(['month']).agg({'Timestamp': 'count'})

# Format dfs as HTML with styling
days_per_month_html = days_per_month_df.to_html(classes='workout-table', index=True, border=0)
workouts_by_month_html = workouts_by_month_df.to_html(classes='workout-table', index=True, border=0)

In [ ]:
# HTML front end code for above 
"""     <!-- Table Section -->
    <div class="section flex-container">
        <div class="plot-box">
            <h2>Workouts by Month and Year</h2>
            {{ workouts_by_month_html | safe }}
        </div>
        <div class="plot-box">
            <h2>Days Exercised per Month</h2>
            {{ days_per_month_html | safe }}
        </div>
    </div> """

In [ ]:
### Code to create last 3 workouts df to put into html table ###

def create_sql_engine():
    # Create a SQLAlchemy engine to connect to the SQLite database
    Database = "habits.db" 
    engine = create_engine(f"sqlite:///{Database}")
    return engine

engine = create_sql_engine()

# Read workouts data from database
with engine.connect() as connection:
    l_3_workouts = pd.read_sql_query(text("""SELECT DISTINCT ha.entry_id, he.log_date, ha.question, ha.answer
                                        FROM habit_entries he
                                        LEFT JOIN habit_answers ha 
                                            ON he.id = ha.entry_id 
                                        WHERE he.habit_id = 3
                                        AND he.log_date IN (
                                                SELECT DISTINCT he2.log_date
                                                FROM habit_entries he2
                                                LEFT JOIN habit_answers ha2
                                                    ON he2.id = ha2.entry_id 
                                                WHERE he2.habit_id = 3
                                                ORDER BY he2.log_date DESC
                                                LIMIT 3)"""), connection)

entry_ids = l_3_workouts['entry_id'].unique().tolist()

# Temp df
columns = ['entry_id', '10RM', 'comment', 'effort', 'reps', 'sets', 'weight', 'workout_type']
workout_df_total = pd.DataFrame(columns=columns)

for entry_id in entry_ids:
    temp1 = l_3_workouts[l_3_workouts['entry_id'] == entry_id]

    temp2 = temp1.pivot(index='entry_id', columns='question', values='answer').reset_index()

    temp2['log_date'] = temp1['log_date'].iloc[0]

    workout_df_total = pd.concat([workout_df_total, temp2], ignore_index=True)

workout_df_total = workout_df_total[["log_date", "workout_type", "weight", "sets", "reps", "effort", "comment"]]

### KPI Statistics

In [ ]:
from datetime import datetime
import pandas as pd

# Get today's date
today = pd.Timestamp.today()

# Get this month and last month as Periods
this_month = today.to_period('M')
last_month = (today - pd.DateOffset(months=1)).to_period('M')
last_month_name = (today - pd.DateOffset(months=1)).strftime('%B')

# Since I just want to get straight count numbers for this db, I can use the shape funciton
count_this_month = aw_final[(aw_final['month'] == this_month) & (aw_final['metric'] == 'Duration')].shape[0]
count_last_month = aw_final[(aw_final['month'] == last_month) & (aw_final['metric'] == 'Duration')].shape[0]
count_total_year = aw_final[(aw_final['StartDate'].dt.year >= 2025) & (aw_final['metric'] == 'Duration')].shape[0]

# Gather workout time metrics
workout_time = aw_final[(aw_final['metric'] == 'Duration')].groupby(['week_period'])['value'].agg(Time='sum', n='count').reset_index()
workout_time = workout_time.tail(5) # Last 3 months of workout data/different in production
workout_time_avg = (workout_time["Time"].mean()/60).round(2)

In [ ]:
# Gather names for above statistics
current_month = today.month
import pandas as pd

today = pd.Timestamp.today()
last_month_name = (today - pd.DateOffset(months=1)).strftime('%B')
print(last_month_name)  # e.g., "May"

In [ ]:
def Read_Apple_Workouts():
    engine = create_engine(f"sqlite:///habits.db")

    # Read from SQLite and parse those columns as datetime
    with engine.connect() as connection:
        apple = pd.read_sql_query(
            text("""SELECT type, value, startDate, endDate 
                FROM apple_data_raw
                WHERE type = 'StepCount' AND value IS NOT NULL and startDate >= '2025-01-01'
                ORDER BY startDate DESC"""), connection,
            parse_dates=['startDate', 'endDate'])
    engine.dispose()

    return apple

apple = Read_Apple_Workouts()

# Group by date (not datetime)
steps_day = apple.groupby(apple['startDate'].dt.date)['value'].agg(steps='sum', n='count').reset_index()

# Convert 'startDate' to datetime
steps_day['startDate'] = pd.to_datetime(steps_day['startDate'])

# Extract month as a period for correct ordering
steps_day['month_period'] = steps_day['startDate'].dt.to_period('M')

# Create a string label for display
steps_day['month_label'] = steps_day['month_period'].dt.strftime('%b %Y')

# Set 'month_label' as a categorical(factor variable) with order based on 'month_period'
month_order = steps_day.sort_values('month_period')['month_label'].unique()
steps_day['month_label'] = pd.Categorical(steps_day['month_label'], categories=month_order, ordered=True)

In [ ]:
today = pd.Timestamp.today() 

three_mon_ago = (today - pd.DateOffset(months=3)).normalize() # Normalize sets the time to midnight

new_df = steps_day[steps_day["startDate"] >= three_mon_ago]

new_df["steps"].mean().round(0).astype("int")

In [ ]:
# Total distances bike, run, and swimming
# Note: There's a difference between counting exercises by duration vs distance because indoor bike workouts don't track distance

miles_total = (aw_final[(aw_final['activity'].isin(['Running', 'Cycling'])) & (aw_final['metric'].str.contains("Distance")) & (aw_final['month'] >= "2025-03")]
    .groupby(['activity'])['value']
    .agg(Total_Miles='sum', n='count')  # compute both mean and count  
    .round(2) # Round to 2 decimal places
    .reset_index())

swim_yards_total = (aw_final[(aw_final['activity'] == "Swimming") & (aw_final['metric'].str.contains("Distance")) & (aw_final['month'] >= "2025-03")]
    .groupby(['activity'])['value']
    .agg(Total_Yards='sum', n='count')  # compute both mean and count
    .round(2) # Round to 2 decimal places
    .reset_index())

In [ ]:
# workouts per week
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(f"sqlite:///habits.db")

# Read from SQLite and parse those columns as datetime
with engine.connect() as connection:
    apple = pd.read_sql_query(
        text("""select aw.StartDate, aw.activity, aw.metric, aw.measurement_type, aw.value from apple_workouts aw 
                where aw.metric = 'Duration' and aw.StartDate > '2025-01-01'
                order by aw.StartDate desc"""), connection,
        parse_dates=['startDate', 'endDate'])
engine.dispose()

# Graveyard

## Old matlotlib and seaborn visuals

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import io
import base64

def generate_matlotlib(): 
    plt.figure(figsize=(8, 4))
    x = range(10)
    y = [i**2 for i in x]
    plt.plot(x, y, marker='o')
    plt.title('Static Plot: Squares')
    plt.xlabel('x')
    plt.ylabel('y')

    # Save static plot to bytes
    static_bytes = io.BytesIO()
    plt.savefig(static_bytes, format='png')
    static_bytes.seek(0)
    static_base64 = base64.b64encode(static_bytes.read()).decode('utf-8')
    matplotlib_plot_url = f"data:image/png;base64,{static_base64}"
    plt.close()

    return matplotlib_plot_url

def generate_seaborn():
    sns.set_theme(style="ticks", palette="pastel")

    # Load the example tips dataset
    tips = sns.load_dataset("tips")

    # Draw a nested boxplot to show bills by day and time
    plt.figure(figsize=(8, 4))  # Optional: ensure consistent sizing
    sns.boxplot(x="day", y="total_bill",
                hue="smoker", palette=["m", "g"],
                data=tips)
    sns.despine(offset=10, trim=True)

    # Save static plot to bytes
    static_bytes = io.BytesIO()
    plt.savefig(static_bytes, format='png')
    static_bytes.seek(0)
    static_base64 = base64.b64encode(static_bytes.read()).decode('utf-8')
    seaborn_plot_url = f"data:image/png;base64,{static_base64}"
    plt.close()

    return seaborn_plot_url

##### old date parsing info may be helpful

In [ ]:
df['dayofweek'] = df.index.dayofweek
#We use the map function along with a lambda function to apply the strftime method to each datetime object in the index. 
# This approach is necessary because the strftime method is designed to work with individual datetime objects, not with entire Series or DataFrames

df['dayofweekchar'] = df.index.map(lambda x: x.strftime('%A'))  # Full day name, e.g., "Monday" 
# Define the desired order of days of the week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
# Convert 'day_of_week_char' to a categorical variable with the specified order
df['dayofweekchar'] = pd.Categorical(df['dayofweekchar'], categories=day_order, ordered=True)

df['quarter'] = df.index.quarter
df['month'] = df.index.month
df['month_char'] = df.index.map(lambda x: x.strftime('%b'))  # Full month name, e.g., "September"

# Define the custom order for the month names
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
# Convert 'month_char' to a categorical data type with the custom order
df['month_char'] = pd.Categorical(df['month_char'], categories=month_order, ordered=True)
    # Sort the DataFrame based on the custom order

df['year'] = df.index.year

#turns it not to numeric for whatever reasosn
df['dayofyear'] = df.index.dayofyear
df['dayofyear'] = pd.to_numeric(df['dayofyear'])

df['dayofmonth'] = df.index.day
df['weekofyear'] = df.index.isocalendar().week

#### Cool XML parsing that I won't end up using ####

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

# Load and parse the file
tree = ET.parse('apple/export.xml') 
root = tree.getroot()

# Extract data from <Record> tags
records = []
for record in root.findall('Record'):
    records.append(record.attrib)

# Convert to DataFrame
df = pd.DataFrame(records)

#df = df[df["sourceName"] == "Ryan’s Apple Watch"]

In [ ]:
(df["type"]).unique()

In [ ]:
filtered_df = df[df["type"] == "HKQuantityTypeIdentifierPhysicalEffort"]